In [5]:
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────
RAW_DIR     = r"../data/raw"
PROCESS_DIR = r"../data/processed"
OUTPUT_PATH = os.path.join(PROCESS_DIR, "merged_dataset.csv")

os.makedirs(PROCESS_DIR, exist_ok=True)

In [6]:
# ── Cell 2: Load all raw CSVs ─────────────────────────────

all_files = [
    f for f in os.listdir(RAW_DIR)
    if f.endswith(".csv")
]
print(f"Found {len(all_files)} source files:")
for f in sorted(all_files):
    path = os.path.join(RAW_DIR, f)
    rows = sum(1 for _ in open(path)) - 1
    print(f"  {f:<35} {rows:>5} rows")

Found 10 source files:
  run_1.csv                             600 rows
  run_100_portscan.csv                  600 rows
  run_2.csv                             600 rows
  run_3.csv                             600 rows
  run_4.csv                             600 rows
  run_5.csv                             600 rows
  run_5_ddos_synth.csv                  600 rows
  run_6.csv                             600 rows
  run_7.csv                             300 rows
  run_8.csv                             500 rows


In [7]:
# ── Cell 3: Merge ─────────────────────────────────────────

df_list = []
for file in all_files:
    df_temp = pd.read_csv(os.path.join(RAW_DIR, file))
    df_temp["source_file"] = file
    df_list.append(df_temp)

df = pd.concat(df_list, ignore_index=True)
print(f"After merge: {df.shape[0]:,} rows × {df.shape[1]} columns")

After merge: 5,600 rows × 29 columns


In [8]:
# ── Cell 4: Sort ──────────────────────────────────────────
# attack_type → phase order (onset→sustained→decay) → timestamp

phase_map          = {'onset': 0, 'sustained': 1, 'decay': 2}
df['_phase_order'] = df['system_phase'].map(phase_map)

df = (df
      .sort_values(['attack_type', '_phase_order', 'timestamp'])
      .drop(columns='_phase_order')
      .reset_index(drop=True))

print("Sorted by: attack_type → phase → timestamp")
print(f"Index reset. Shape: {df.shape}")

Sorted by: attack_type → phase → timestamp
Index reset. Shape: (5600, 29)


In [9]:
# ── Cell 5: Validate ──────────────────────────────────────

errors = []

if df.isnull().sum().sum() != 0:
    errors.append(f"Nulls found: {df.isnull().sum().sum()}")

if len(df) != 5600:
    errors.append(f"Row count mismatch: expected 5600, got {len(df)}")

expected_attacks = {'none', 'ddos', 'portscan', 'crypto'}
if set(df['attack_type'].unique()) != expected_attacks:
    errors.append(f"Unexpected attack types: {df['attack_type'].unique()}")

expected_phases = {'onset', 'sustained', 'decay'}
if set(df['system_phase'].unique()) != expected_phases:
    errors.append(f"Unexpected phases: {df['system_phase'].unique()}")

if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print("All validation checks passed ✓")
    print(f"  Rows     : {len(df):,}")
    print(f"  Columns  : {df.shape[1]}")
    print(f"  Nulls    : 0")
    print(f"  Attacks  : {sorted(df['attack_type'].unique())}")
    print(f"  Phases   : {sorted(df['system_phase'].unique())}")

All validation checks passed ✓
  Rows     : 5,600
  Columns  : 29
  Nulls    : 0
  Attacks  : ['crypto', 'ddos', 'none', 'portscan']
  Phases   : ['decay', 'onset', 'sustained']


In [10]:
# ── Cell 6: Summary ───────────────────────────────────────

print("── Attack type distribution ──")
print(df['attack_type'].value_counts().to_string())

print("\n── Phase × Attack type ──")
crosstab          = pd.crosstab(df['attack_type'], df['system_phase'])
crosstab          = crosstab[['onset', 'sustained', 'decay']]
crosstab['TOTAL'] = crosstab.sum(axis=1)
print(crosstab.to_string())

print("\n── Source files ──")
print(df['source_file'].value_counts().to_string())

print("\n── Column dtypes ──")
print(df.dtypes.to_string())

── Attack type distribution ──
attack_type
none        3900
ddos         600
portscan     600
crypto       500

── Phase × Attack type ──
system_phase  onset  sustained  decay  TOTAL
attack_type                                 
crypto           64        113    323    500
ddos            120        360    120    600
none            862       1748   1290   3900
portscan        143        284    173    600

── Source files ──
source_file
run_5_ddos_synth.csv    600
run_1.csv               600
run_5.csv               600
run_2.csv               600
run_3.csv               600
run_4.csv               600
run_100_portscan.csv    600
run_6.csv               600
run_8.csv               500
run_7.csv               300

── Column dtypes ──
timestamp                      object
run_id                          int64
cpu_user_pct                  float64
cpu_system_pct                float64
cpu_idle_pct                  float64
context_switch_rate           float64
interrupt_rate                f

In [11]:
# ── Cell 6: Summary ───────────────────────────────────────

print("── Attack type distribution ──")
print(df['attack_type'].value_counts().to_string())

print("\n── Phase × Attack type ──")
crosstab          = pd.crosstab(df['attack_type'], df['system_phase'])
crosstab          = crosstab[['onset', 'sustained', 'decay']]
crosstab['TOTAL'] = crosstab.sum(axis=1)
print(crosstab.to_string())

print("\n── Source files ──")
print(df['source_file'].value_counts().to_string())

print("\n── Column dtypes ──")
print(df.dtypes.to_string())

── Attack type distribution ──
attack_type
none        3900
ddos         600
portscan     600
crypto       500

── Phase × Attack type ──
system_phase  onset  sustained  decay  TOTAL
attack_type                                 
crypto           64        113    323    500
ddos            120        360    120    600
none            862       1748   1290   3900
portscan        143        284    173    600

── Source files ──
source_file
run_5_ddos_synth.csv    600
run_1.csv               600
run_5.csv               600
run_2.csv               600
run_3.csv               600
run_4.csv               600
run_100_portscan.csv    600
run_6.csv               600
run_8.csv               500
run_7.csv               300

── Column dtypes ──
timestamp                      object
run_id                          int64
cpu_user_pct                  float64
cpu_system_pct                float64
cpu_idle_pct                  float64
context_switch_rate           float64
interrupt_rate                f

In [12]:
# ── Cell 7: Save ──────────────────────────────────────────

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")
print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.2f} MB")

Saved → ../data/processed\merged_dataset.csv
File size: 0.89 MB
